In [1]:
from pathlib import Path
import json
import time

from faster_whisper import WhisperModel
from tqdm.auto import tqdm

d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path.cwd().parent / "L30_data"

VIDEO_DIR = DATA_DIR / "videos"

OUTPUT_DIR = DATA_DIR / "subtitles"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_paths = sorted(VIDEO_DIR.glob("*.mp4"))

print(f"Found {len(video_paths)} videos")

Found 96 videos


In [3]:
MODEL_NAME = "large-v3-turbo"

DEVICE = "cuda"

COMPUTE_TYPE = "float16"

model = WhisperModel(
    MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

print(f"Model loaded on {DEVICE} with {COMPUTE_TYPE}.")

Model loaded on cuda with float16.


In [4]:
def transcribe_video(video_path: Path):

    output_path = OUTPUT_DIR / f"{video_path.stem}.json"

    if output_path.exists():
        return

    segments, info = model.transcribe(
        str(video_path),
        language="vi",
        vad_filter=True,
    )

    subtitles = []

    for seg in segments:

        subtitles.append(
            {
                "start": round(seg.start, 3),
                "end": round(seg.end, 3),
                "text": seg.text.strip(),
            }
        )

    with open(output_path, "w", encoding="utf-8") as f:

        json.dump(
            subtitles,
            f,
            ensure_ascii=False,
            indent=2,
        )

In [5]:
start = time.time()

for video_path in tqdm(video_paths):

    transcribe_video(video_path)

print(f"Done in {(time.time()-start)/60:.2f} minutes")

100%|██████████| 96/96 [15:26<00:00,  9.66s/it]

Done in 15.45 minutes
